In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# mengatur gaya visualisasi seaborn
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# 1. Memuat Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/aamir28/student-sleep-screen-time-and-academic-performance/student_sleep_screen_time_academic_performance.csv")
df

In [ ]:
print(f"Ukuran dataset: {df.shape[0]} baris, {df.shape[1]} kolom")

In [ ]:
df.info()

In [ ]:
df.describe()

# 2. Cek Duplikasi

In [ ]:
df.duplicated()

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Jumlah baris duplikat: {duplicate_count}")

**Tidak ada baris yang terduplikasi di dataset**. Jika ada data duplikat, sebaiknya dihapus (di-drop).

In [ ]:
if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Baris duplikat telah dihapus.")
else:
    print("Tidak ada data duplikat.")

# 3. Cek Missing Value

In [ ]:
df.isnull()

In [ ]:
missing_val = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df) * 100)
missing_df = pd.DataFrame(
    {'Jumlah Missing': missing_val,
     'Persentase (%)': missing_percent}
)

print("Kolom yang ada missing value:")
display(missing_df[missing_df['Jumlah Missing'] > 0])

# 4. Imputasi Missing Value

In [ ]:
# untuk kolom numerik, bisa diisi dengan median
numeric_cols_with_na = [
    'sleep_quality_score',
    'energy_level',
    'concentration_difficulty'
]


for col in numeric_cols_with_na:
    if col in df.columns and df[col].isnull().sum() > 0:
        
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        
        print(f"Imputasi kolom {col} menggunakan median: {median_val}")

# untuk kolom kategorikal, bisa diisi dengan modus
cat_cols_with_na = ['device_free_before_bed']
for col in cat_cols_with_na:
  if col in df.columns and df[col].isnull().sum() > 0:
      
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
      
    print(
        f"Imputasi kolom kategorikal '{col}' menggunakan modus: {mode_val}"
    )

print(
    '\nSisa missing value setelah imputasi:',
    df.isnull().sum().sum(),
    '\n',
)

In [ ]:
# Kolom CGPA adalah target variable, bisa di-drop saja
df.dropna(inplace=True)

print(
    '\nSisa missing value:',
    df.isnull().sum().sum(),
    '\n',
)

# 5. Membetulkan Fitur yang Bermasalah

Fitur gender masih memiliki value yang beragam

In [ ]:
df['gender'].unique()

In [ ]:
GENDERS = {
    'Male': 'Male',
    'male': 'Male',
    'M': 'Male',
    'Female': 'Female',
    'female': 'Female',
    'F': 'Female',
    'Prefer not to say': 'Prefer not to say',
    'Non-binary': 'Non-binary'
}

df['gender_fix'] = df['gender'].map(GENDERS)

df

In [ ]:
# Menghapus kolom gender yang lama
df.drop(columns=['gender'], inplace=True)

# 6. Deteksi dan Penanganan Outlier

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(numeric_cols)

In [ ]:
# Visualisasi Boxplot sebelum penanganan outlier untuk beberapa kolom
plt.figure(figsize=(30, 6))
sns.boxplot(data=df[['avg_sleep_hours_weekday', 'weekend_sleep_delta_hours', 'daily_screen_time_hours']])
plt.title('Boxplot untuk Deteksi Outlier')
plt.show()

In [ ]:
# Menghitung IQR dan menangani outlier (misalnya dengan metode capping / clipping)
for col in ['daily_screen_time_hours', 'weekly_study_hours']:
  if col in numeric_cols:
    print(f"Menangani outlier di kolom {col}")
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    print(f"Q1 : {Q1}")
    print(f"Q2 : {Q3}")
    print(f"IQR: {IQR}")
    print(f"Lower bound : {lower_bound}")
    print(f"Upper bound : {upper_bound}")
      
    outliers_count = df[
        (df[col] < lower_bound) | (df[col] > upper_bound)
    ].shape[0]
    print(f"Kolom '{col}': ditemukan {outliers_count} outlier.")

    # Capping nilai outlier agar batasnya mentok di lower/upper bound
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

    print("=========================================================")


# 7. Analisis Korelasi

In [ ]:
plt.figure(figsize=(12, 10))
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(
    correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5,
    vmin=-1.0, vmax=1.0
)
plt.title('Matriks Korelasi Fitur Numerik')
plt.show()

# 8. Analisis Bivariat

In [ ]:
# Contoh 1: Hubungan antara daily_screen_time_hours vs cgpa (Scatter Plot dengan garis regresi)
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x='sleep_quality_score',
    y='cgpa',
    hue='gender_fix',
    alpha=0.7,
    palette='Set2',
)
sns.regplot(
    data=df,
    x='sleep_quality_score',
    y='cgpa',
    scatter=False,
    color='black',
)
plt.title('Bivariat: Sleep Quality Score vs CGPA')
plt.show()

In [ ]:
# Contoh 1: Hubungan antara daily_screen_time_hours vs cgpa (Scatter Plot dengan garis regresi)
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x='sleep_quality_score',
    y='avg_sleep_hours_weekday',
    # hue='gender_fix',
    alpha=0.7,
    # palette='Set2',
)
sns.regplot(
    data=df,
    x='sleep_quality_score',
    y='avg_sleep_hours_weekday',
    scatter=False,
    color='black',
)
plt.title('Bivariat: Sleep Quality Score vs Average Sleep Hours Weekday')
plt.show()

# 

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='major_category', y='cgpa')
plt.title('Bivariat: Kategori Jurusan vs CGPA')
plt.xticks(rotation=45)
plt.show()

# 9. Analisis Multivariat (Pairplot)

In [ ]:
# Memilih beberapa fitur numerik utama agar plot mudah dibaca
multivar_cols = [
    'cgpa',
    'daily_screen_time_hours',
    'avg_sleep_hours_weekday',
    'weekly_study_hours',
]
sns.pairplot(
    df[multivar_cols + ['gender_fix']].dropna(),
    hue='gender_fix',
    diag_kind='kde',
    palette='husl',
)
plt.suptitle('Analisis Multivariat (Pairplot Fitur Utama)', y=1.02)
plt.show()


# 10. Cek Skewness

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
skewness_values = df[numeric_cols].skew()

skew_df = pd.DataFrame(
    {'Skewness': skewness_values, 'Tipe Distribusi': ''}
)
for idx, val in skewness_values.items():
  if val > 1:
    skew_df.loc[idx, 'Tipe Distribusi'] = 'High Positive Skew'
  elif val < -1:
    skew_df.loc[idx, 'Tipe Distribusi'] = 'High Negative Skew'
  else:
    skew_df.loc[idx, 'Tipe Distribusi'] = 'Relatif Normal'

print(skew_df, '\n')

**NOTE: MODEL BIASANYA CUMA MAU "MAKAN" DATA YANG BERDISTRIBUSI NORMAL, KALO SKEW GK ENAK KATANYA**
<br>
Bisa melakukan transformasi log untuk "menormalkan" datanya.

In [ ]:
df['pre_sleep_screen_time_min_log'] = np.log1p(df['pre_sleep_screen_time_min'])

In [ ]:
print(f"Before: {df['pre_sleep_screen_time_min'].skew()}")
print(f"After: {df['pre_sleep_screen_time_min_log'].skew()}")

walah, malah makin jadi skew negatif cik 😹 (emot kucing ketawa)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(df['pre_sleep_screen_time_min'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Pre-Sleep Screen Time (data asli)')
axes[0].set_xlabel('Menit')
axes[0].set_ylabel('Frekuensi')

sns.histplot(df['pre_sleep_screen_time_min_log'], kde=True, ax=axes[1], color='skyblue')
axes[1].set_title('Pre-Sleep Screen Time (log transform)')
axes[1].set_xlabel('Nilai log')
axes[1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

mari kita coba pakai fungsi akar untuk menormalisasi distribusi datanya

In [ ]:
df['pre_sleep_screen_time_min_sqrt'] = np.sqrt(df['pre_sleep_screen_time_min'])

In [ ]:
print(f"Data asli: {df['pre_sleep_screen_time_min'].skew()}")
print(f"Log-transform: {df['pre_sleep_screen_time_min_log'].skew()}")
print(f"Fungsi akar: {df['pre_sleep_screen_time_min_sqrt'].skew()}")

hmmmm better lah ya

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['pre_sleep_screen_time_min'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Pre-Sleep Screen Time (data asli)')
axes[0].set_xlabel('Menit')
axes[0].set_ylabel('Frekuensi')

sns.histplot(df['pre_sleep_screen_time_min_log'], kde=True, ax=axes[1], color='skyblue')
axes[1].set_title('Pre-Sleep Screen Time (log transform)')
axes[1].set_xlabel('Nilai log')
axes[1].set_ylabel('Frekuensi')

sns.histplot(df['pre_sleep_screen_time_min_sqrt'], kde=True, ax=axes[2], color='skyblue')
axes[2].set_title('Pre-Sleep Screen Time (sqrt)')
axes[2].set_xlabel('Nilai sqrt(menit)')
axes[2].set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

NICE!

# 11. Encoding Categorical Features

In [ ]:
df.select_dtypes(include='object')

In [ ]:
df.drop(columns=['student_id'], inplace=True) # student id dihapus aja, kenapa ya...?

## 11.1. Nominal

Fitur kategorikal nominal adalah 'major_category', 'residence_type', 'chronotype', 'primary_device', 'semester_type', 'device_free_before_bed', 'has_part_time_job', dan 'gender_fix'. KEBETULAN, memang semuanya nominal.*
<br>
*Apakah ada yang berinterpretasi lain?
<br>
Umumnya, tipe data nominal di-encode dengan metode One Hot Encoding

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(cat_cols)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encoded_array = encoder.fit_transform(df[cat_cols])

In [ ]:
encoded_array.shape

In [ ]:
encoded_cols = encoder.get_feature_names_out(cat_cols)
encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=df.index)

df_dropped = df.drop(columns=cat_cols)

df_final = pd.concat([df_dropped, encoded_df], axis=1)

In [ ]:
df_final

CURSE OF DIMENSIONALITY!!!

## 11.2. Ordinal

KEBETULAN, datasetnya ga ada kategorikal yang ordinal, ordinalnya udah dalam bentuk numerik semua. Jadi udah aman. Ada yang bisa sebutkan ordinalnya itu mana aja?

# 12. Penskalaan Fitur

In [ ]:
df_final.agg(['min', 'max', 'mean', 'std']).T

Lihat guys nilai min sama max nya, beda beda ga sih untuk tiap fitur??? that's why kita butuh **feature scaling** biar skalanya tuh sama! Kenapa harus di-scaling???? Ada yg bisa jawab?

## MIN MAX SCALER

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler_minmax = MinMaxScaler()
df_minmax_scaled = df_final.copy()
df_minmax_scaled = scaler_minmax.fit_transform(df_final)
 
print('\nHasil Normalisasi (MinMaxScaler - Rentang 0 sampai 1):')
display(df_minmax_scaled[:3])
print(
    'Nilai Minimum:\n', df_minmax_scaled.min()
)  # Harus 0
print(
    'Nilai Maksimum:\n', df_minmax_scaled.max()
)  # Harus 1

## Standard Scaler

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler_standard = StandardScaler()
df_standard_scaled = df_final.copy()
df_standard_scaled = scaler_standard.fit_transform(df_final)
 
print('\nHasil Normalisasi (StandardScaler):')
display(df_standard_scaled[:3])
print(
    'Nilai Minimum:\n', df_standard_scaled.min()
)  # Harus 0
print(
    'Nilai Maksimum:\n', df_standard_scaled.max()
)  # Harus 1

# 13. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df_final.drop(columns=['cgpa'])
y = df_final['cgpa']

scaler_minmax = MinMaxScaler()
X = scaler_minmax.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
print(f"Ukuran X_train: {X_train.shape}")
print(f"Ukuran y_train: {y_train.shape}")
print(f"Ukuran X_test : {X_test.shape}")
print(f"Ukuran y_test : {y_test.shape}")

# Latihan

1. Berdasarkan *pipeline* di atas, ada indikasi *data leakage* ketika *pipeline* tersebut diterapkan ke tahap pelatihan model. Apakah kamu bisa menunjukkan bagian mana yang dapat menyebabkan *data leakage*? Jelaskan alasannya!
   -
   
Jawab: 

Penyebab: penanganan outlier (IQR), imputasi missing value (median, mean), transformasi fitur skeww (log, sqrt), featur encoding (standart, minmax)

Alasan: tahapan pre process itu dilakukan pada dataset sebelum pembagian/split data train dan test. Jadinyastatistik data uji (median, mea, log, sqrt, dll itu) ikut terhitung dan bocor ke training model.

2. Apa solusi agar *pipeline* tersebut bebas dari *data leakage*?
   -
Jawab: 

Karna data leaking itu terjadi karen split belakangan, berarti solusinya lakukan split data testing dan data training dulu, lalu pengolahan/perhitungan statistikanya belakangan.

In [ ]:
plt.figure(figsize=(12, 10))
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(
    correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5,
    vmin=-1.0, vmax=1.0
)
plt.title('Matriks Korelasi Fitur Numerik')
plt.savefig("/kaggle/working/corr_matrix.png")
plt.show()

3. Perhatikan matriks korelasi fitur di atas! Bagaimana interpretasimu terhadap matriks korelasi berikut?
   -
Jawab: 

Korelasi terkuat yang meningkatkan cgpa seorang mahasiswa adalah nilai tidur dan rata rata tidur di akhir pekan: sleep_quality_score (r = 0.47) dan avg_sleep_hours_weekday (r = 0.44) 

Dan yang berpengaruh terkuat menurunkan cgpa adalah screen time sebelum tidur: pre_sleep_screen_time_min (r = -0.22)

In [ ]:
cols = ['avg_sleep_hours_weekday', 'weekend_sleep_delta_hours', 'daily_screen_time_hours']

plt.figure(figsize=(14, 7))
ax = sns.boxplot(data=df[cols])
plt.title('Boxplot untuk Deteksi Outlier dengan Anotasi Statistik', fontsize=14, pad=15)
plt.ylabel('Nilai / Jam')

for i, col in enumerate(cols):
    mean_val = df[col].mean()
    median_val = df[col].median()
    min_val = df[col].min()
    max_val = df[col].max()

    stats_text = (
        f"Mean: {mean_val:.2f}\n"
        f"Median: {median_val:.2f}\n"
        f"Min: {min_val:.2f}\n"
        f"Max: {max_val:.2f}"
    )
    
    # Menempatkan teks di atas masing-masing boxplot
    # i adalah posisi indeks sumbu X (0, 1, 2)
    # max_val * 1.05 digunakan agar teks berada sedikit di atas titik maksimum/whisker
    ax.text(
        x=i, 
        y=df[col].max() * 1.05, 
        s=stats_text, 
        horizontalalignment='center', 
        verticalalignment='bottom', 
        fontsize=9,
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round,pad=0.5'))

plt.tight_layout()
plt.savefig("/kaggle/working/boxplot.png")
plt.show()

https://www.geeksforgeeks.org/data-analysis/box-plot/

4. Apa interpretasimu terhadap grafik *boxplot* di atas?
   -
Jawab:

avg_sleep_hours_weekday: terlihat ada titik kecil di bawah garis, berarti adanya data outlier rendah, yaitu mahasiswa yang durasi tidur hariannya sekitar 1.5 jam

weekend_sleep_delta_hours: rentang garisnya pendek dari kotak, berarti sebaran jam tidur akhir pekan dekat-dekat median. 

daily_screen_time_hours: ini ga ada outlier, tapi rentang datanya luas (sekitar dari 1 sampai 11)

5. Load ulang dataset dan simpan ke variabel 'df_tugas'!
   -

In [ ]:
import pandas as pd
import numpy as np

df_tugas = pd.read_csv('/kaggle/input/datasets/aamir28/student-sleep-screen-time-and-academic-performance/student_sleep_screen_time_academic_performance.csv')

In [ ]:
df_tugas = pd.read_csv('/kaggle/input/datasets/aamir28/student-sleep-screen-time-and-academic-performance/student_sleep_screen_time_academic_performance.csv')

6. Pisahkan kolom menjadi fitur $X_{tugas}$ dan target $y_{tugas}$!
   -

In [ ]:
df_tugas = df_tugas.dropna(subset=['cgpa'])

X = df_tugas.drop(columns=['cgpa'])
y = df_tugas['cgpa']

7. Buat data latih dan data uji dengan proporsi $75:25$ dan random_state=42 serta shuffle=True ($75\%$ data latih dan $25\%$ data uji)! Simpan di variabel X_train_tugas, X_test_tugas, y_train_tugas, dan y_test_tugas!
   -

In [ ]:
from sklearn.model_selection import train_test_split

X_train_tugas, X_test_tugas, y_train_tugas, y_test_tugas = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)

8. Lakukan *exploratory data analysis* pada df_tugas: fitur mana saja yang memiliki *missing value*?
   -

In [ ]:
missing_info = df_tugas.isnull().sum()
print("Kolom dengan missing value:\n", missing_info[missing_info > 0])

9. Lakukan *data preprocessing* pada data latih: lakukan imputasi pada kolom yang memiliki *missing value*?
   -

In [ ]:
from sklearn.impute import SimpleImputer

num_cols = X_train_tugas.select_dtypes(include=[np.number]).columns
imputer_num = SimpleImputer(strategy='median')
X_train_tugas[num_cols] = imputer_num.fit_transform(X_train_tugas[num_cols])

cat_cols = X_train_tugas.select_dtypes(include=['object', 'category']).columns
if len(cat_cols) > 0:
    imputer_cat = SimpleImputer(strategy='most_frequent')
    X_train_tugas[cat_cols] = imputer_cat.fit_transform(X_train_tugas[cat_cols])

Hint: gunakan SimpleImputer dari Scikit-Learn untuk melakukan fit_transform imputasi pada data latih! Jika kesulitan, cukup gunakan cara seperti di atas.
<br>
*many roads lead to Roma~~~*

10. Lakukan EDA pada df_tugas: fitur mana saja yang berkorelasi kuat dengan cgpa?
    -
   Jawab: 

Fitur yang berkorelasi paling kuat positif dengan cgpa adalah sleep_quality_score (r = 0.47) dan avg_sleep_hours_weekday (r = 0.44)  

In [ ]:
train_df = X_train_tugas.copy()
train_df['cgpa'] = y_train_tugas
corr = train_df.corr(numeric_only=True)['cgpa'].sort_values(ascending=False)
print("Korelasi fitur dengan CGPA:\n", corr)

11. Lakukan EDA pada df_tugas: apakah ada multikolinearitas pada data latih? Jelaskan alasannya!
    -
    Jawab: 

Ya, terdapat multikolinearitas. Alasannya gabungan avg_sleep_hours_weekday dan sleep_quality_score memiliki nilai korelasi linier tinggi (r = 0.74). Hubungan linear tinggi bisa berarti redudansi informasi dan membuat model kurang stabil.

In [ ]:
corr_matrix = X_train_tugas[num_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [column for column in upper_tri.columns if any(upper_tri[column] > 0.70)]
print("Fitur dengan indikasi multikolinearitas tinggi (>0.70):", high_corr)

12. Lakukan EDA pada df_tugas: fitur mana saja yang mengalami *skewness*?
    -
    Jawab:
    
pre_sleep_screen_time_min
caffeine_drinks_per_day
naps_per_week
sleep_quality_score
weekly_study_hours

In [ ]:
skewness = X_train_tugas[num_cols].skew()
print("Nilai Skewness Fitur:\n", skewness)
skewed_cols = skewness[abs(skewness) > 0.5].index.tolist()
print("Fitur yang mengalami skewness (|skew| > 0.5):", skewed_cols)

13. Lakukan EDA pada df_tugas: sebutkan fitur nominal dan fitur ordinal!
    -
    Jawab:

Fitur nominal (tidak memiliki urutan/tingkatan) : gender, gender_fix, device_free_before_bed 

Fitur ordinal (memiliki tingkatan/skala bertingkat): year_of_study, energy_level, concentration_difficulty

In [ ]:
# Tulis kode di sini . . .

14. Lakukan *data preprocessing* pada data latih: lakukan transformasi untuk fitur yang *skew*!
    -

In [ ]:
for col in skewed_cols:
    X_train_tugas[col] = np.log1p(X_train_tugas[col])

15. Lakukan *data preprocessing* pada data latih: lakukan *encoding* pada data kategorikal!
    -

In [ ]:
from sklearn.preprocessing import OneHotEncoder

if 'gender' in X_train_tugas.columns:
    X_train_tugas['gender'] = X_train_tugas['gender'].replace({
        'male': 'Male', 'M': 'Male', 'female': 'Female', 'F': 'Female'
    })

encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encoded_cat = encoder.fit_transform(X_train_tugas[cat_cols])

encoded_cat_df = pd.DataFrame(encoded_cat, columns=encoder.get_feature_names_out(cat_cols), index=X_train_tugas.index)
X_train_tugas = X_train_tugas.drop(columns=cat_cols).join(encoded_cat_df)

16. Lakukan *data preprocessing* pada data latih: lakukan penskalaan fitur!
    -

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_tugas)

X_train_tugas_final = pd.DataFrame(X_train_scaled, columns=X_train_tugas.columns, index=X_train_tugas.index)
print("Prapemrosesan Data Latih Selesai!")

17. Bonus Challenge: lakukan *data preprocessing* pada data uji!
    -
    Hint: gunakan fit_transform() pada data latih dan transform() pada data uji.

In [ ]:
# Jawab: . . .
# Tulis kode di sini . . .